In [2]:
#!/usr/bin/env python3
import os
from openai import OpenAI
import json
import requests
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv('OPENAI_APIKEY'))

data = {
    "cpu" : "98%",
    "memory" : "80%",
    "disk" : "90%",
    "network" : "70%",
    "timestamp" : "2023-06-01T12:00:00Z"
}

jdata =json.dumps(data)

print("Sending data to OpenAI API..."+jdata)


Sending data to OpenAI API...{"cpu": "98%", "memory": "80%", "disk": "90%", "network": "70%", "timestamp": "2023-06-01T12:00:00Z"}


In [17]:
def get_server_health(server_id :str) ->str:
    """ Returns CPU usage , memory usage
        for a given server id.
    """
    print(f"-> Getting server health for server id: {server_id}")
    
    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }
    
    result = metrics.get(server_id, {"cpu": "N/A", "memory": "N/A", "status": "Unknown"})
    return json.dumps(result)
    #return result
    

In [18]:
get_server_health("payment-server-01")

-> Getting server health for server id: payment-server-01


'{"cpu": "98%", "memory": "40%", "status": "Warning"}'

In [19]:
def fetch_recent_logs(server_id: str, lines: int =8)->str:
    """ Returns the last N lines of logs 
    """
    print(f"-> Fetching last {lines} lines of logs for server id: {server_id}")
    
    
    #Different log scenarios for different servers to trigger diffrent agnet behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]   
    }
    
    # default logs if server_id not found in specific list
    default_logs = [
        "[INFO] System stable", 
        "[INFO] Heartbeat signal received"
    ]
    
    logs= log_database.get(server_id, default_logs)
    
    return json.dumps({"logs": logs[-lines:]})

In [ ]:
# ---- TASK 1 : Implement the Restart tool ----
def restart_service(server_id : str) -> str:
    """ Simulate restarting a serverice on serever 
        return a message indicating the restart status.
    """
    print(f"-> TOOL : Restarting service .... on server id: {server_id}")
    
    response= {
        "status" : "success",
        "message" : "Server restarted successfully"
    } 
    
    return json.dumps(response)

    
    

In [21]:
# ---- TASK 2 : Implement the Escalate tool ----

def escalate_to_engineer( summary : str ) -> str :
    """ 
    ### TODO: Implement the function to escalate the issue to a human engineer. 
    """
    response= {
        "status": "Escalated",
        "message": "The issue has been escalated to a human engineer for further investigation."
    }
    
    return json.dumps(response)

In [23]:
# Map of available functions for the agent to call  
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer
}



In [24]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 3: Define Schema for restart_service ---
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Simulates restarting a service on the specified server to address high CPU or memory usage.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server to restart."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- >>>> TASK 4: Define Schema for escalate_to_engineer ---
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary": {"type": "string", "description": "A summary of the issue, findings, and any attempted fixes."}
                },
                "required": ["summary"]
            }
        }
    }
]

In [ ]:
def run_it_agent(user_issue: str):
    """ 
    This function simulates an AI agent that can diagnose and resolve server issues based on user input.
    """
    print(f" --> New Incident Reported: {user_issue}")
    
    messages = [
        {"role": "system", "content": " Your a level 1 IT Support agent. Investigate Server issues."
                                       " If CPU or Memory is >90 % , restart the service. If logs show critical dependency errors (like connection refused), that restart won't fix the issue, escalate to a human engineer. If the issue is unknown, escalate to a human engineer."
        },
        {"role": "user", "content": user_issue}
    ]
    
    while True:
        print ("-> Sending request to OpenAI API...")
        print("\n [AI is thinking...]")
        response = client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = messages,
            tools = tools_schema,
            tool_choice="auto",
        )
        
        response_message = response.choices[0].message
        messages.append(response_message)
        
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # Retrieve the corresponding function from AVAILABLE_FUNCTIONS
                function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
                
                if function_to_call:
                    # Call the function and get the result
                    tool_result = function_to_call(**function_args)
                    
                else:
                    tool_result = json.dumps({"error": f"Unknown tool: {function_name}"})

                # Match this result to the model's original tool call.
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": tool_result,
                })
        else:
            print(response_message.content)
            return response_message.content
